# 1. Main results: Table 1 and the main-text figures

This reproduces the benchmark comparison at the center of the paper. It runs the four
main experiments and the ablations on the real data, builds Table 1, and draws the
main-text figures.

Runtime is about two hours on 20 cores. To check that the pipeline runs end to end
before committing to the full job, lower `--replications-main` (for example to 5) and
set `--n-jobs` to your core count; the numbers will be noisy but the plumbing is the
same.


## Step 1 - run the experiments

`--all` runs Experiments 1-4, the ablations, and the fidelity study in one pass.
The flags below reproduce the paper's replication counts: 500 for the main comparison,
100 for the adaptive and ACS studies, 200 for the ablations. Results and a
`run_manifest.json` (the exact configuration, per-file code hashes, and data sources)
are written to `results_main/`.


In [ ]:
import subprocess
cmd = [
    'python', 'run_experiments.py', '--all',
    '--replications-adaptive', '100', '--replications-acs', '100',
    '--replications-ablation', '200',
    '--use-real-data', '--ihdp-npz-path', 'data/raw/ihdp_npci_1-100.merged.npz', '--twins-csv-path', 'data/raw/twins.csv', '--acic-dir', 'data/raw/acic', '--lalonde-csv-path', 'data/raw/lalonde.csv',
    '--results-dir', 'results_main', '--figures-dir', 'results_main/figs_raw',
    '--n-jobs', '20',
]
subprocess.run(cmd, cwd='..', check=True)


## Step 2 - build Table 1

`generate_paper_artifacts.py` reads the summary CSVs and writes the LaTeX table the
paper includes. Each cell is `RMSE / coverage` at the given privacy budget; the table
is at epsilon = 1. This is the source of the main results table in the paper.


In [ ]:
subprocess.run(
    'python generate_paper_artifacts.py --results-dir results_main --out-dir auto'.split(),
    cwd='..', check=True)
print(open('../auto/table1.tex').read())


## Step 3 - draw the figures

`plot_all_figures` regenerates every figure from the summary CSVs. The workload-
dimension figure has a second curve for the SNR-thresholded pipeline, which needs the
extra run in notebook 2; you can pass that directory here once it exists, or leave it
for now and redraw later.


In [ ]:
from pathlib import Path
import sys; sys.path.insert(0, '..')
import plot_figures
plot_figures.plot_all_figures(Path('../results_main'), Path('../figures'))


The figure files map to the paper as follows. The label keys match the `\label`
commands in the LaTeX source, so you can cross-reference regardless of float numbering.

| File | Content | Label |
|---|---|---|
| `fig1_ate_rmse_comparison.pdf` | ATE RMSE vs privacy budget | `fig:exp1_rmse` |
| `fig3_coverage_comparison.pdf` | 95% CI coverage vs budget | `fig:exp2_coverage` |
| `fig4_ci_length_comparison.pdf` | CI length (log scale) | `fig:exp2_length` |
| `fig5_adaptive_vs_fixed.pdf` | Causal-AIM vs fixed workload | `fig:exp3_adaptive` |
| `fig2_ate_bias_comparison.pdf` | Mean absolute ATE error | `fig:exp1_bias` |
| `fig6_acs_rmse_heatmap.pdf` | ACS RMSE by n and budget | `fig:exp4_acs_rmse` |
| `fig7_acs_coverage.pdf` | ACS coverage by n and budget | `fig:exp4_acs_coverage` |


In [ ]:
# Preview one figure inline.
from IPython.display import IFrame
IFrame('../figures/fig1_ate_rmse_comparison.pdf', width=760, height=520)
